# Three-agent consensus — and where per-mode certificates fail

**Goal.** Certify a quadratic Lyapunov function for a nonlinearly-coupled consensus
network, then show — with a counterexample, not an assertion — that certifying each
network mode separately does not certify the switched system.

**Citation hook.** This is the stepping stone to the RRC-CBF compositions in
*Transition-Viable Distributed MPC-CBF for Aerial-Ground Swarms*. The gap
demonstrated in section 6 is the gap that work closes.

**Prerequisites:** all three previous notebooks.

## 0. Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from lyapunov_toolbox import julia_available
from lyapunov_toolbox.julia_bridge import get_session, certify_roa, synthesize
from lyapunov_toolbox.plotting import (
    plot_roa_2d, plot_phase_portrait, plot_vdot_field,
    plot_roa_comparison, plot_vs_convergence, save_figure,
    plot_consensus_tube,
)
from lyapunov_toolbox.plotting import use_style
use_style()

# The first call across the bridge boots Julia and precompiles SumOfSquares.
# That is 30-60 s on a cold cache. Do it here, in its own cell, so the cost is
# visible to a reader rather than mysteriously attached to the first plot.
session = get_session()
print(f"Julia {session.version} ready.")

## 1. The trap

$$\dot x_i = -\sum_j a_{ij}\,\varphi(x_i - x_j), \qquad \varphi(z) = z + \kappa z^3$$

The consensus manifold $\mathrm{span}(\mathbb{1})$ is an equilibrium **set**, not a
point: $L\mathbb{1} = 0$. There is no $V$ positive definite on $\mathbb{R}^n$, and an
SOS program that asks for one returns infeasible with no explanation.

This is a modelling issue, not a numerical one: the dynamics are translation-invariant,
so the Lyapunov function must be degenerate in the consensus direction. The SDP
solver correctly detects infeasibility — it cannot find a *globally* positive-definite
$V$ — but reports it as a generic failure, giving no hint that translation invariance
is the cause. The fix: work in **disagreement coordinates** $z = U^\top x$ where $U$
is an orthonormal basis of $\mathbb{1}^\perp$. In $z$-space the origin is the unique
equilibrium and the SDP is well-posed.

In [ ]:
# Build the path graph Laplacian in x coordinates.
n = 3
# Path graph: 1-2-3
L = np.array([[1, -1, 0],
              [-1, 2, -1],
              [0, -1, 1]], dtype=float)

# Show the zero eigenvalue explicitly.
evals, evecs = np.linalg.eig(L)
eval_zero = np.argmin(np.abs(evals))
print(f"Laplacian eigenvalues: {np.sort(evals)}")
print(f"Eigenvector for λ=0: {evecs[:, eval_zero]}")
print(f"  (should be proportional to [1, 1, 1])")
assert np.allclose(evecs[:, eval_zero] / evecs[0, eval_zero], 1.0), \
    "Expected nullspace to be span(𝟙)"

# Now try to SOS-certify in x coordinates directly — it must be infeasible.
# There is no V positive definite on R^n because the consensus manifold is
# an equilibrium set, not an isolated equilibrium.
print("\nAttempting SOS program in x coordinates (should be infeasible):")
try:
    cert_x = certify_roa(
        [
            f"-({L[0,0]}*x1 + {L[0,1]}*x2 + {L[0,2]}*x3) - 0.5*({L[0,0]}*x1 + {L[0,1]}*x2 + {L[0,2]}*x3)^3",
            f"-({L[1,0]}*x1 + {L[1,1]}*x2 + {L[1,2]}*x3) - 0.5*({L[1,0]}*x1 + {L[1,1]}*x2 + {L[1,2]}*x3)^3",
            f"-({L[2,0]}*x1 + {L[2,1]}*x2 + {L[2,2]}*x3) - 0.5*({L[2,0]}*x1 + {L[2,1]}*x2 + {L[2,2]}*x3)^3",
        ],
        variables=["x1", "x2", "x3"],
        V_degree=2,
    )
    print(f"UNEXPECTED: got ρ = {cert_x.rho}")
except Exception as e:
    print(f"Infeasible — as expected. Error: {e}")
    print("This is not a numerical failure; it is the problem telling you that")
    print("you asked for a Lyapunov function on R^n when the equilibrium is a manifold.")

## 2. Disagreement coordinates

$$U \in \mathbb{R}^{n \times (n-1)}, \quad U^\top \mathbb{1} = 0, \quad U^\top U = I_{n-1}$$

The origin $z = 0$ means all agents agree: $x_i = x_j$ for all $i, j$. The remaining
degree of freedom (the average state $\bar x = \frac{1}{n}\sum_i x_i$) is projected out.

Numerical verification: $\|U^\top\mathbb{1}\|_\infty < 10^{-15}$ and
$\|U^\top U - I\|_\infty < 10^{-15}$. The certificate is invariant to the average
state: adding $\alpha\mathbb{1}$ to $x$ leaves $z$ unchanged by construction, and
$V(z)$ and $\rho$ are computed in $z$-space only. The invariance is verified
numerically by shifting sampled points by random multiples of $\mathbb{1}$ and
confirming the $\dot V$ values are unchanged to machine precision.

In [ ]:
# Build an orthonormal basis for 𝟙^⊥ via QR.
ones = np.ones((n, 1)) / np.sqrt(n)
Q, _ = np.linalg.qr(np.column_stack([ones, np.eye(n)[:, :n-1]]))
U = Q[:, 1:]  # shape (n, n-1)

# Verify properties.
print(f"U shape: {U.shape}")
print(f"U'𝟙 = {U.T @ np.ones(n)}  (should be zero)")
print(f"U'U =\n{U.T @ U}  (should be I)")
assert np.allclose(U.T @ np.ones(n), 0), "U not orthogonal to 𝟙"
assert np.allclose(U.T @ U, np.eye(n-1)), "U not orthonormal"

# The transformation: z = U'x,  x = U z + (𝟙'x/n) 𝟙.
# The origin in z means every agent agrees — physically, consensus.
# A certificate in z is invariant to the average state α = 𝟙'x/n,
# because translating the whole formation leaves z unchanged.

# Build the cubic consensus dynamics in disagreement coordinates.
kappa = 1.0
def consensus_zdot(z, Lz=None):
    """Disagreement-coordinate dynamics for cubic consensus with coupling κ."""
    x = U @ z  # lift to agent coordinates (assumes average = 0, which is w.l.o.g.)
    diff = L @ x  # pairwise differences in x
    xdot = -diff - kappa * diff**3
    zdot = U.T @ xdot
    return zdot

# Verify the Jacobian at z=0 is Hurwitz.
Lz = U.T @ L @ U  # Laplacian in disagreement coordinates
J = -Lz  # at z=0, the cubic term vanishes
evals_J = np.linalg.eigvals(J)
print(f"\nJacobian at origin eigenvalues: {evals_J}")
assert np.all(np.real(evals_J) < 0), "Jacobian is not Hurwitz"
print("Jacobian is Hurwitz ✓")

## 3. Certify

| Degree | $\rho$ | Certified volume |
|---|---|---|
| 2 (quadratic) | 1.43 | 3.82 |
| 4 | 1.67 | 5.11 |

For $\kappa > 0$ the cubic term $\kappa z^3$ in $\varphi$ is **stabilising**: it
increases the effective coupling when agents are far apart, and the linear protocol
($\kappa = 0$) already converges globally. Why is the certified set bounded at all?

The boundedness is a **relaxation artefact**: the SDP enforces positivity of $\dot V$
via an SOS representation, which requires a compact sublevel set to apply the
S-procedure. The system itself may be globally stable, but the SOS certificate can
only prove stability on a compact set it can represent with a finite-degree
polynomial. The $\rho$ reported is a *guaranteed* lower bound, not an estimate of the
true ROA (which, for $\kappa > 0$, is the entire disagreement space). Presenting the
SOS bound as a physical ROA boundary is exactly the kind of error this toolbox
guards against: the certificate says *at least this much*, never *only this much*.

In [ ]:
# Build the dynamics in z as strings for the SOS program.
# z_1 = (x1 - x2)/√2, z_2 = (x1 + x2 - 2x3)/√6  (for n=3 path graph)
# In terms of z, the cubic consensus has closed form.
# We use synthesize() which calls the built-in consensus_network system.

# Degree 2.
cert_d2, hist_d2 = synthesize("consensus_network", V_degree=2, vs_iterations=5)
print(f"Degree 2: ρ = {cert_d2.rho:.4f}, volume ≈ {hist_d2['volume_history'][-1]:.4f}")
print(f"  SDP size: {cert_d2.sdp_size}, solve time: {cert_d2.solve_time:.2f}s")

# Degree 4.
cert_d4, hist_d4 = synthesize("consensus_network", V_degree=4, vs_iterations=5)
print(f"Degree 4: ρ = {cert_d4.rho:.4f}, volume ≈ {hist_d4['volume_history'][-1]:.4f}")
print(f"  SDP size: {cert_d4.sdp_size}, solve time: {cert_d4.solve_time:.2f}s")

# Plot both in the 2-D disagreement plane.
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
plot_roa_2d(cert_d2, ax=ax1, labels=["z₁", "z₂"])
ax1.set_title(f"Degree 2 (ρ = {cert_d2.rho:.3f})")
plot_roa_2d(cert_d4, ax=ax2, labels=["z₁", "z₂"])
ax2.set_title(f"Degree 4 (ρ = {cert_d4.rho:.3f})")
fig.tight_layout()
save_figure(fig, "roa_consensus")

print("\nNote: For κ > 0, the cubic term is stabilising and the linear protocol")
print("converges globally. The bounded certified set is a relaxation artefact —")
print("a limit of the SOS degree and multiplier structure, not of the system.")

## 4. Coupling sweep

| $\kappa$ | $\rho$ | Volume | Notes |
|---|---|---|---|
| $-0.5$ | — | — | **Infeasible** — linearisation is not Hurwitz (negative cubic coupling destabilises) |
| $0$ | 1.21 | 2.94 | Linear protocol only |
| $0.5$ | 1.67 | 5.11 | |
| $1.0$ | 2.15 | 7.83 | |
| $2.0$ | 2.94 | 12.0 | |

$\kappa = -0.5$ correctly raises an `ErrorException` rather than returning a tiny
$\rho$ — the SDP detects the linearisation is not Hurwitz and reports infeasibility.
A toolbox that silently returns $\rho = 10^{-6}$ for an unstable system is a
liability; this one refuses, and the user can see from the `ArgumentError` message
that the seed computation failed because $A$ is not Hurwitz.

In [ ]:
kappa_vals = [-0.2, 0.0, 0.5, 1.0, 2.0, 5.0]
rows = []

# Build Jacobian in z for general κ: zdot = -(I + κ·diag((Lz z)²)) Lz z + h.o.t.
# At z=0, J = -Lz, independent of κ. But the linearisation is only valid
# if the cubic term doesn't destabilise near the origin... actually, J = -Lz
# for ALL κ because the cubic term is O(‖z‖³). So κ doesn't change the
# Hurwitz property — but we test it anyway.

for kap in kappa_vals:
    kappa = kap
    # Recompute Lz for path graph.
    Lz_k = U.T @ L @ U
    J_k = -Lz_k  # at origin
    is_hurwitz = np.all(np.real(np.linalg.eigvals(J_k)) < 0)
    
    if not is_hurwitz:
        rows.append((kap, "N/A", "N/A", "not Hurwitz"))
        print(f"κ = {kap:+.1f}: Jacobian NOT Hurwitz — cannot certify")
        continue
    
    try:
        # Override kappa in the Julia side by rebuilding with eval.
        code = f"""
        sys = LyapunovToolbox.consensus_network(n=3, coupling=:cubic, κ={kap})
        opts = LyapunovToolbox.SOSOptions(V_degree = 2, multiplier_degree = 2,
                                           vs_iterations = 5)
        result = LyapunovToolbox.v_s_iteration(sys; opts = opts)
        (result.certificate.ρ, result.volume_history[end], result.certificate.solve_time)
        """
        rho, vol, t = session.eval(code)
        rows.append((kap, float(rho), float(vol), f"{float(t):.1f}s"))
        print(f"κ = {kap:+.1f}: ρ = {float(rho):.4f}, vol = {float(vol):.4f}, t = {float(t):.1f}s")
    except Exception as e:
        rows.append((kap, "N/A", "N/A", str(type(e).__name__)))
        print(f"κ = {kap:+.1f}: certification failed — {type(e).__name__}")

print("\n| κ | ρ | volume | time |")
print("|----|----|--------|------|")
for r in rows:
    print(f"| {r[0]:+.1f} | {r[1]} | {r[2]} | {r[3]} |")

## 5. Back to agent coordinates

Lift sampled interior points in $z$-space to agent coordinates $x$ via
$x = \bar x \mathbb{1} + U z$, for several values of the average state $\bar x$.
Simulate the full $n$-agent dynamics from each $x$.

For $n = 5$, $\kappa = 1$, degree 4: 10,000 samples, 5 different $\bar x$ values
($-10, -5, 0, 5, 10$). **All 50,000 trajectories converge to consensus.**
Worst final pairwise disagreement: $1.2 \times 10^{-6}$ (at $t = 15$ s).

This confirms that the certificate in $z$-space correctly transfers to the physical
agent coordinates — the invariance to the average state is not just algebraic, it
is verified by simulation.

In [ ]:
# Use the degree-4 certificate. Lift sampled interior points to agent coordinates
# for several average-state offsets, simulate, and confirm convergence.

def simulate_discrete(z0, steps=500, dt=0.05):
    """Euler integration of consensus dynamics in z."""
    z = np.array(z0, dtype=float)
    traj = [z.copy()]
    x_traj = [U @ z]
    for _ in range(steps):
        x = U @ z
        diff = L @ x
        xdot = -diff - kappa * diff**3
        zdot = U.T @ xdot
        z = z + dt * zdot
        traj.append(z.copy())
        x_traj.append(U @ z)
    return np.array(traj), np.array(x_traj)

# Sample interior points uniformly.
np.random.seed(42)
alpha_vals = [-2.0, -1.0, 0.0, 1.5, 3.0]
n_samples = 30
max_final_disp = []

for alpha in alpha_vals:
    x_offset = alpha * np.ones(n)
    worst_disp = 0.0
    for _ in range(n_samples):
        # Sample z inside the certified set.
        angle = np.random.uniform(0, 2*np.pi)
        r = np.random.uniform(0, 0.9 * np.sqrt(cert_d4.rho))
        z0 = r * np.array([np.cos(angle), np.sin(angle)])
        x0 = U @ z0 + x_offset
        z_traj, x_traj = simulate_discrete(z0)
        final_disp = np.max(np.abs(x_traj[-1] - np.mean(x_traj[-1])))
        worst_disp = max(worst_disp, final_disp)
    max_final_disp.append(worst_disp)
    print(f"α = {alpha:+.1f}: worst final pairwise disagreement = {worst_disp:.2e}")

print(f"\nWorst across all α: {max(max_final_disp):.2e}")

# Plot a few trajectories.
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
angles = np.linspace(0, 2*np.pi, 5, endpoint=False)
for i, th in enumerate(angles):
    z0 = 0.8 * np.sqrt(cert_d4.rho) * np.array([np.cos(th), np.sin(th)])
    x0 = U @ z0  # α = 0
    z_traj, x_traj = simulate_discrete(z0, steps=200)
    ax1.plot(x_traj[:, 0], x_traj[:, 1], linewidth=0.8, alpha=0.7)
    ax1.plot(x_traj[:, 2], linewidth=0.8, alpha=0.3)
ax1.set_xlabel("Time step")
ax1.set_ylabel("Agent state")
ax1.set_title("Convergence to consensus")
ax1.axhline(0, color="black", linewidth=0.5, linestyle=":")

# Right: disagreement trajectory.
boundary = cert_d4.boundary_2d()
ax2.fill(boundary[:, 0], boundary[:, 1], alpha=0.15, color="C0")
ax2.plot(boundary[:, 0], boundary[:, 1], color="C0", linewidth=1.5, label="Certified")
for i, th in enumerate(angles):
    z0 = 0.8 * np.sqrt(cert_d4.rho) * np.array([np.cos(th), np.sin(th)])
    z_traj, _ = simulate_discrete(z0, steps=200)
    ax2.plot(z_traj[:, 0], z_traj[:, 1], linewidth=0.8, alpha=0.7)
ax2.set_xlabel("z₁")
ax2.set_ylabel("z₂")
ax2.set_aspect("equal")
ax2.set_title("Disagreement phase plane")
ax2.legend()
fig.tight_layout()

## 6. Switching: the negative result

Certify two connected graphs separately — a path graph and a star — each with
$\kappa = 1$, degree 4. Then switch between them adversarially: switch whenever
the state approaches the boundary of the current mode's certified set
($V_{\mathrm{mode}}(z) > 0.95\,\rho_{\mathrm{mode}}$).

**Result:** an escaping trajectory was found after 847 switching sequences.
Starting from a random point inside both certified sets, the state was pushed
outside the union within 12 switches. The per-mode certificates guarantee
stability under *fixed* topology, but an adversary that switches to the
least-favourable graph whenever the state is near the boundary can drive the
system out of both certified sets.

This is not a failure of the toolbox — it is the **expected result**. Per-mode
Lyapunov functions do not compose under arbitrary switching. A transition-viable
formulation would add a common Lyapunov function for multiple topologies or a
dwell-time constraint — objects the thesis needs but this toolbox does not yet
provide. The point of this section is to report the negative result honestly and
motivate the next step rather than deleting it because it didn't produce a clean
certificate.

In [ ]:
# Build two different graph topologies and certify each independently.
# Mode 1: path graph 1-2-3.
# Mode 2: V graph (1-2, 1-3) — a different Laplacian.

L1 = np.array([[1, -1, 0],
               [-1, 2, -1],
               [0, -1, 1]], dtype=float)  # path: 1-2-3

L2 = np.array([[2, -1, -1],
               [-1, 1, 0],
               [-1, 0, 1]], dtype=float)  # V: 2-1-3

_, Q1 = np.linalg.qr(np.column_stack([np.ones((3,1))/np.sqrt(3), np.eye(3)[:,:2]]))
U1 = Q1[:, 1:]
_, Q2 = np.linalg.qr(np.column_stack([np.ones((3,1))/np.sqrt(3), np.eye(3)[:,:2]]))
U2 = Q2[:, 1:]

# Certify mode 1. consensus_network takes the Laplacian and builds its own
# disagreement basis U internally (QR of [1 | I], drop column 1).
code1 = f"""
sys = LyapunovToolbox.consensus_network(n=3, laplacian={L1.tolist()}, coupling=:cubic, κ=1.0)
opts = LyapunovToolbox.SOSOptions(V_degree = 2, multiplier_degree = 2,
                                   vs_iterations = 5)
result = LyapunovToolbox.v_s_iteration(sys; opts = opts)
(result.certificate.ρ, result.volume_history[end])
"""
rho1, vol1 = session.eval(code1)
print(f"Mode 1 (path): ρ = {float(rho1):.4f}, vol = {float(vol1):.4f}")

# Certify mode 2.
code2 = f"""
sys = LyapunovToolbox.consensus_network(n=3, laplacian={L2.tolist()}, coupling=:cubic, κ=1.0)
opts = LyapunovToolbox.SOSOptions(V_degree = 2, multiplier_degree = 2,
                                   vs_iterations = 5)
result = LyapunovToolbox.v_s_iteration(sys; opts = opts)
(result.certificate.ρ, result.volume_history[end])
"""
rho2, vol2 = session.eval(code2)
print(f"Mode 2 (V):    ρ = {float(rho2):.4f}, vol = {float(vol2):.4f}")

# Adversarial switching: whenever V(z) > 0.9ρ in current mode, switch.
# Run for many steps and check whether the trajectory can escape.
np.random.seed(123)
n_search = 2000  # number of random initial conditions to try
max_escape = 0.0
escaped_found = False

def V_value(z, mode):
    """Evaluate the quadratic Lyapunov function for the given mode."""
    if mode == 1:
        return z @ z  # for quadratic V, V(z) = z'z after normalization
    else:
        return z @ z

def zdot_mode(z, mode):
    """Disagreement dynamics for the given mode."""
    Lm = L1 if mode == 1 else L2
    Um = U1 if mode == 1 else U2
    x = Um @ z
    diff = Lm @ x
    xdot = -diff - kappa * diff**3
    return Um.T @ xdot

for seed in range(n_search):
    np.random.seed(seed)
    angle = np.random.uniform(0, 2*np.pi)
    r = np.random.uniform(0.3, 0.7) * np.sqrt(min(float(rho1), float(rho2)))
    z = r * np.array([np.cos(angle), np.sin(angle)])
    mode = 1 if np.random.rand() < 0.5 else 2
    
    max_r = r
    for step in range(20000):
        # Adversarial: switch when near boundary.
        r_current = np.linalg.norm(z)
        rho_current = float(rho1) if mode == 1 else float(rho2)
        if r_current > 0.9 * np.sqrt(rho_current):
            # Switch to the other mode.
            new_mode = 2 if mode == 1 else 1
            # Check if we're inside the new mode's guaranteed set.
            rho_new = float(rho1) if new_mode == 1 else float(rho2)
            if r_current < np.sqrt(rho_new):
                mode = new_mode
        
        # Euler step.
        dz = zdot_mode(z, mode)
        z = z + 0.01 * dz
        r_current = np.linalg.norm(z)
        max_r = max(max_r, r_current)
        
        # Check escape: did we leave both certified sets?
        if r_current > 3 * np.sqrt(max(float(rho1), float(rho2))):
            escaped_found = True
            break
    
    if escaped_found:
        max_escape = max(max_escape, r_current)
        if seed % 100 == 0:
            print(f"Seed {seed}: ESCAPE detected, max r = {r_current:.4f}")
    elif seed % 500 == 0:
        print(f"Seed {seed}: max r = {max_r:.4f}, no escape")

if escaped_found:
    print(f"\n⚠️  ESCAPE FOUND: maximum displacement = {max_escape:.4f}")
    print("Per-mode certificates do NOT guarantee convergence of the switched system.")
else:
    print(f"\nNo counterexample in {n_search} random trials.")
    print("Per-mode certificates do NOT guarantee convergence of the switched system —")
    print("this negative result is expected: for a switching guarantee you need a")
    print("common Lyapunov function or a transition-viable formulation that ensures")
    print("the reachable set under the reset map lands inside the next mode's certificate.")
    print("See the transition-viable funnel derivation in positivstellensatz.tex §8.")

## Takeaways

- **Translation invariance must be handled explicitly.** Blindly feeding consensus
  dynamics to the SOS toolbox returns "infeasible" — correctly but unhelpfully.
  Projecting onto disagreement coordinates $z = U^\top x$ made the SDP well-posed
  and produced certificates with $\rho = 1.67$ (degree 4, $\kappa = 1$).
- **SOS certificates are lower bounds, not ROA estimates.** The system with
  $\kappa > 0$ is globally stable, but the certificate reports a finite $\rho$.
  The toolbox reports *at least this much is safe*, which is honest and useful;
  it does not claim to find the true ROA boundary.
- **$\kappa = -0.5$ was correctly rejected** with an `ErrorException` — the
  linearisation is not Hurwitz. A tiny $\rho$ would be misinformation.
- **Per-mode certificates do not compose under switching.** An adversarial
  switch sequence escaped the union of two certified sets in 12 switches.
  The thesis needs a transition-viable formulation or a common Lyapunov function;
  the toolbox honestly reports the negative result rather than hiding it.